<a href="https://colab.research.google.com/github/Shahul187/aml-alert-triage/blob/main/notebooks/08_tableau_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np

base = '/content/drive/MyDrive/aml-project/'
a = pd.read_csv(base + 'scored_alerts.csv')

a['score_rank'] = a['model_score'].rank(ascending=False, method='first')
a['score_pctile'] = (a['score_rank'] / len(a) * 100).round(2)
a['review_band'] = pd.cut(a['score_pctile'],
    bins=[0, 5, 10, 20, 30, 50, 100],
    labels=['Top 5%','5-10%','10-20%','20-30%','30-50%','Bottom 50%'])
a['outcome'] = np.where(a['Is_laundering'] == 1, 'Laundering', 'Clean')
a['payment_channel'] = np.where(a['is_cash'] == 1, 'Cash', 'Electronic')
a['flow'] = np.where(a['is_crossborder'] == 1, 'Cross-border', 'Domestic')
a['month'] = pd.to_datetime(a['Date']).dt.to_period('M').astype(str)

out = a[['Date','month','Amount','payment_channel','flow','Laundering_type',
         'outcome','Is_laundering','model_score','score_pctile','review_band',
         'in_top_5pct','hybrid_flag']]
out.to_csv(base + 'tableau_alerts.csv', index=False)

print(f"{len(out):,} rows, {out.shape[1]} columns")
print(out['review_band'].value_counts().sort_index())

Mounted at /content/drive
56,544 rows, 13 columns
review_band
Top 5%         2828
5-10%          2827
10-20%         5654
20-30%         5655
30-50%        11308
Bottom 50%    28270
Name: count, dtype: int64
